## Model Enhancement Phase
Building on the validated baseline (02_modeling.ipynb). This notebook covers:
1. Fair Random Forest comparison (matched model complexity + overfitting check)
2. Blind-spot-targeted feature engineering
3. Probability calibration
4. Hyperparameter tuning (Optuna)

In [1]:
import joblib
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

lgb_model = joblib.load(r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\fraud_model_full.pkl")
X_held_out, y_held_out = joblib.load(r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\held_out_data.pkl")

feature_columns = list(X_held_out.columns)
pr_auc_lgb = average_precision_score(y_held_out, lgb_model.predict_proba(X_held_out)[:, 1])
print(f"LightGBM PR-AUC (held_out): {pr_auc_lgb:.4f}")

LightGBM PR-AUC (held_out): 0.7198


In [6]:
X_train, y_train = joblib.load(r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\train_data.pkl")
print(X_train.shape, y_train.shape)

(1311225, 17) (1311225,)


## Fair Random Forest Comparison

Checking whether the earlier Random Forest result (PR-AUC 0.8352) was real or just
overfitting from a deeper, less constrained model than LightGBM was allowed to use.

In [7]:
# Original (unfair) Random Forest — rebuild it here since this is a fresh kernel
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf_model.fit(X_train, y_train)

pr_auc_rf_train = average_precision_score(y_train, rf_model.predict_proba(X_train)[:, 1])
pr_auc_rf_held = average_precision_score(y_held_out, rf_model.predict_proba(X_held_out)[:, 1])
print(f"Original RF (depth=10) — TRAIN PR-AUC:    {pr_auc_rf_train:.4f}")
print(f"Original RF (depth=10) — HELD-OUT PR-AUC: {pr_auc_rf_held:.4f}")

# Matched-complexity Random Forest — the fair comparison against LightGBM
rf_matched = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=100,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf_matched.fit(X_train, y_train)

pr_auc_rf_matched_train = average_precision_score(y_train, rf_matched.predict_proba(X_train)[:, 1])
pr_auc_rf_matched_held = average_precision_score(y_held_out, rf_matched.predict_proba(X_held_out)[:, 1])
print(f"\nMatched RF (depth=5) — TRAIN PR-AUC:      {pr_auc_rf_matched_train:.4f}")
print(f"Matched RF (depth=5) — HELD-OUT PR-AUC:   {pr_auc_rf_matched_held:.4f}")
print(f"\nLightGBM — HELD-OUT PR-AUC:                {pr_auc_lgb:.4f}")

Original RF (depth=10) — TRAIN PR-AUC:    0.9189
Original RF (depth=10) — HELD-OUT PR-AUC: 0.8352

Matched RF (depth=5) — TRAIN PR-AUC:      0.8485
Matched RF (depth=5) — HELD-OUT PR-AUC:   0.8199

LightGBM — HELD-OUT PR-AUC:                0.7198


In [8]:
# Check the actual imbalance ratio in your training data
n_legit = (y_train == 0).sum()
n_fraud = (y_train == 1).sum()
true_ratio = n_legit / n_fraud
print(f"Legit: {n_legit}, Fraud: {n_fraud}, True imbalance ratio: {true_ratio:.1f}")

# Re-run RF with class_weight explicitly matched to LightGBM's scale_pos_weight=20,
# instead of sklearn's "balanced" (which uses the TRUE ratio, likely much higher)
rf_weight_matched = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=100,
    class_weight={0: 1, 1: 20},   # matches LightGBM's scale_pos_weight=20 exactly
    random_state=42, n_jobs=-1,
)
rf_weight_matched.fit(X_train, y_train)

pr_auc_rf_wm_train = average_precision_score(y_train, rf_weight_matched.predict_proba(X_train)[:, 1])
pr_auc_rf_wm_held = average_precision_score(y_held_out, rf_weight_matched.predict_proba(X_held_out)[:, 1])
print(f"\nWeight-matched RF (class_weight={{0:1,1:20}}) — TRAIN PR-AUC:    {pr_auc_rf_wm_train:.4f}")
print(f"Weight-matched RF (class_weight={{0:1,1:20}}) — HELD-OUT PR-AUC: {pr_auc_rf_wm_held:.4f}")

Legit: 1310447, Fraud: 778, True imbalance ratio: 1684.4

Weight-matched RF (class_weight={0:1,1:20}) — TRAIN PR-AUC:    0.7247
Weight-matched RF (class_weight={0:1,1:20}) — HELD-OUT PR-AUC: 0.6044


### Finding Random Forest's Own Best Weight

Instead of assuming LightGBM's weight=20 is fair to both models, letting
Random Forest find its own optimal weighting point.

In [9]:
for w in [20, 50, 100, 200, 500, 1000]:
    rf_w = RandomForestClassifier(
        n_estimators=200, max_depth=5, min_samples_leaf=100,
        class_weight={0: 1, 1: w}, random_state=42, n_jobs=-1,
    )
    rf_w.fit(X_train, y_train)
    score = average_precision_score(y_held_out, rf_w.predict_proba(X_held_out)[:, 1])
    print(f"RF weight={w}: held-out PR-AUC = {score:.4f}")

RF weight=20: held-out PR-AUC = 0.6044
RF weight=50: held-out PR-AUC = 0.7929
RF weight=100: held-out PR-AUC = 0.7968
RF weight=200: held-out PR-AUC = 0.8234
RF weight=500: held-out PR-AUC = 0.8335
RF weight=1000: held-out PR-AUC = 0.8292


### Finding LightGBM's Own Best Weight

Same sweep applied to LightGBM, so the final comparison is each model
at its own best setting — not one model tuned and the other guessed.

In [15]:
SEED = 42

for w in [20, 50, 100, 200, 500, 1000]:
    lgb_w = lgb.LGBMClassifier(
        n_estimators=200, num_leaves=15, max_depth=5, min_child_samples=100,
        reg_alpha=1.0, reg_lambda=1.0,
        scale_pos_weight=w, random_state=SEED, verbose=-1,
    )
    lgb_w.fit(X_train, y_train)
    score = average_precision_score(y_held_out, lgb_w.predict_proba(X_held_out)[:, 1])
    print(f"LightGBM weight={w}: held-out PR-AUC = {score:.4f}")

LightGBM weight=20: held-out PR-AUC = 0.7198
LightGBM weight=50: held-out PR-AUC = 0.2073
LightGBM weight=100: held-out PR-AUC = 0.1628
LightGBM weight=200: held-out PR-AUC = 0.1135
LightGBM weight=500: held-out PR-AUC = 0.0491
LightGBM weight=1000: held-out PR-AUC = 0.0607


### Overfitting Check — RF at weight=500

Confirming the winning configuration generalizes rather than memorizing
the ~778 fraud rows, the same check applied to LightGBM earlier.

In [16]:
rf_500 = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=100,
    class_weight={0: 1, 1: 500}, random_state=SEED, n_jobs=-1,
)
rf_500.fit(X_train, y_train)
train_score = average_precision_score(y_train, rf_500.predict_proba(X_train)[:, 1])
held_score = average_precision_score(y_held_out, rf_500.predict_proba(X_held_out)[:, 1])
print(f"RF weight=500 — TRAIN: {train_score:.4f}, HELD-OUT: {held_score:.4f}, gap: {train_score-held_score:+.4f}")

RF weight=500 — TRAIN: 0.8560, HELD-OUT: 0.8335, gap: +0.0225


### Bootstrap Confidence Intervals

Defining the bootstrap function locally since this notebook has a separate
kernel from 02_modeling.ipynb — resampling the held-out set with replacement
many times to see how much PR-AUC varies by chance alone.

In [19]:
def bootstrap_pr_auc_fast(y_true, probs, n_iterations=200, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    n = len(y_true)
    scores = []
    for _ in range(n_iterations):
        idx = rng.randint(0, n, n)
        scores.append(average_precision_score(y_true[idx], probs[idx]))
    return np.array(scores)

# Predict ONCE per model — this is the only expensive step
probs_rf500 = rf_500.predict_proba(X_held_out)[:, 1]
probs_lgb20 = lgb_model.predict_proba(X_held_out)[:, 1]

ci_rf500 = tuple(np.percentile(bootstrap_pr_auc_fast(y_held_out, probs_rf500), [2.5, 97.5]))
ci_lgb20 = tuple(np.percentile(bootstrap_pr_auc_fast(y_held_out, probs_lgb20), [2.5, 97.5]))

print(f"RF (weight=500) 95% CI:      [{ci_rf500[0]:.4f}, {ci_rf500[1]:.4f}]")
print(f"LightGBM (weight=20) 95% CI: [{ci_lgb20[0]:.4f}, {ci_lgb20[1]:.4f}]")

RF (weight=500) 95% CI:      [0.8085, 0.8546]
LightGBM (weight=20) 95% CI: [0.6877, 0.7492]


### Amount-Stratified Recall — Random Forest

Repeating the same quartile breakdown used for LightGBM, to check whether
RF's overall PR-AUC advantage holds consistently across transaction sizes,
or concentrates in one range.

In [20]:
results_rf = pd.DataFrame({
    "amount": X_held_out["amount"].values,
    "actual_fraud": y_held_out.values,
    "predicted_prob": rf_500.predict_proba(X_held_out)[:, 1],
})

bins = [0, 127091, 441423, 1517771, 10_000_001]
labels = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
results_rf["amount_bucket"] = pd.cut(results_rf["amount"], bins=bins, labels=labels)

T_BLOCK = 0.90
T_REVIEW = 0.0001

def compute_recall(group):
    fraud_rows = group[group["actual_fraud"] == 1]
    if len(fraud_rows) == 0:
        return pd.Series({"n_fraud": 0, "recall_at_block": np.nan, "recall_at_review": np.nan})
    caught_block = (fraud_rows["predicted_prob"] >= T_BLOCK).sum()
    caught_review = (fraud_rows["predicted_prob"] >= T_REVIEW).sum()
    return pd.Series({
        "n_fraud": len(fraud_rows),
        "recall_at_block": caught_block / len(fraud_rows),
        "recall_at_review": caught_review / len(fraud_rows),
    })

recall_by_bucket_rf = results_rf.groupby("amount_bucket", observed=True).apply(compute_recall)
print("Random Forest (weight=500):")
print(recall_by_bucket_rf)

Random Forest (weight=500):
               n_fraud  recall_at_block  recall_at_review
amount_bucket                                            
Q1 (lowest)      200.0         0.365000               1.0
Q2               206.0         0.631068               1.0
Q3               190.0         0.731579               1.0
Q4 (highest)     225.0         0.795556               1.0


### RF Recall at Its Own Cost-Optimal Threshold (0.85)

Re-checking block-recall by amount bucket using RF's actual best threshold
from the cost sweep above, instead of LightGBM's borrowed 0.90.

In [21]:
fraud_probs_rf = rf_500.predict_proba(X_held_out[y_held_out == 1])[:, 1]
fraud_probs_lgb = lgb_model.predict_proba(X_held_out[y_held_out == 1])[:, 1]

print("RF fraud-case probability distribution:")
print(pd.Series(fraud_probs_rf).describe())
print("\nLightGBM fraud-case probability distribution:")
print(pd.Series(fraud_probs_lgb).describe())

RF fraud-case probability distribution:
count    822.000000
mean       0.857385
std        0.184944
min        0.334891
25%        0.849914
50%        0.931411
75%        0.987144
max        0.999048
dtype: float64

LightGBM fraud-case probability distribution:
count     8.220000e+02
mean      8.008923e-01
std       3.804243e-01
min      3.294429e-147
25%       9.913872e-01
50%       1.000000e+00
75%       1.000000e+00
max       1.000000e+00
dtype: float64


In [22]:
def total_cost(y_true, probs, amounts, t_block, cost_false_block=5, cost_false_review_per_unit=1, t_review=0.0001):
    tier = np.where(probs >= t_block, "BLOCK", np.where(probs >= t_review, "REVIEW", "ALLOW"))
    cost = 0
    for i in range(len(y_true)):
        if y_true[i] == 1 and tier[i] == "ALLOW":
            cost += amounts[i]              # missed fraud costs the full amount
        elif y_true[i] == 0 and tier[i] == "BLOCK":
            cost += cost_false_block         # wrongly blocking a legit transaction
        elif tier[i] == "REVIEW":
            cost += cost_false_review_per_unit  # review always has a small fixed cost
    return cost

probs_rf = rf_500.predict_proba(X_held_out)[:, 1]
amounts = X_held_out["amount"].values
y_true = y_held_out.values

print("Finding RF's own cost-optimal BLOCK threshold:")
for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    c = total_cost(y_true, probs_rf, amounts, t_block=t)
    print(f"  t_block={t}: total cost = {c:,.0f}")

Finding RF's own cost-optimal BLOCK threshold:
  t_block=0.5: total cost = 1,239,725
  t_block=0.6: total cost = 1,207,041
  t_block=0.7: total cost = 1,202,977
  t_block=0.75: total cost = 1,202,238
  t_block=0.8: total cost = 1,202,181
  t_block=0.85: total cost = 1,202,119
  t_block=0.9: total cost = 1,202,141
  t_block=0.95: total cost = 1,202,274


### LightGBM Cost Sweep — Same Formula, Direct Comparison

Running the identical cost function used for RF above, on LightGBM, so the
two models' total costs are directly comparable rather than comparing RF's
fresh number against an old $17.2M figure computed differently.

In [23]:
probs_lgb = lgb_model.predict_proba(X_held_out)[:, 1]

print("LightGBM cost at various thresholds (same cost function as RF):")
for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    c = total_cost(y_true, probs_lgb, amounts, t_block=t)
    print(f"  t_block={t}: total cost = {c:,.0f}")

LightGBM cost at various thresholds (same cost function as RF):
  t_block=0.5: total cost = 46,456,259
  t_block=0.6: total cost = 46,455,885
  t_block=0.7: total cost = 46,455,683
  t_block=0.75: total cost = 46,455,615
  t_block=0.8: total cost = 46,455,561
  t_block=0.85: total cost = 46,455,517
  t_block=0.9: total cost = 46,455,419
  t_block=0.95: total cost = 46,455,298


In [24]:
def compute_recall_at_threshold(group, t_block):
    fraud_rows = group[group["actual_fraud"] == 1]
    if len(fraud_rows) == 0:
        return pd.Series({"n_fraud": 0, "recall_at_block": np.nan})
    caught = (fraud_rows["predicted_prob"] >= t_block).sum()
    return pd.Series({"n_fraud": len(fraud_rows), "recall_at_block": caught / len(fraud_rows)})

recall_rf_085 = results_rf.groupby("amount_bucket", observed=True).apply(
    lambda g: compute_recall_at_threshold(g, 0.85)
)
print("Random Forest at t_block=0.85 (its own cost-optimal):")
print(recall_rf_085)

Random Forest at t_block=0.85 (its own cost-optimal):
               n_fraud  recall_at_block
amount_bucket                          
Q1 (lowest)      200.0         0.500000
Q2               206.0         0.640777
Q3               190.0         0.836842
Q4 (highest)     225.0         0.995556


### Blind-Spot Check — Does RF Handle LightGBM's Known Failure Cases Better?

Re-scoring the exact fraud cases that LightGBM missed almost completely
(near-zero probability despite being real fraud) using Random Forest,
to see if this is the root cause of the 39x cost gap.

In [25]:
# Re-identify LightGBM's catastrophic misses on the full held-out set
# (same logic as the original demo_df-based check, now on the full held-out data)
lgb_probs_all = lgb_model.predict_proba(X_held_out)[:, 1]
rf_probs_all = rf_500.predict_proba(X_held_out)[:, 1]

held_out_check = X_held_out.copy()
held_out_check["actual_fraud"] = y_held_out.values
held_out_check["lgb_prob"] = lgb_probs_all
held_out_check["rf_prob"] = rf_probs_all

lgb_catastrophic_misses = held_out_check[
    (held_out_check["actual_fraud"] == 1) & (held_out_check["lgb_prob"] < 0.001)
]

print(f"LightGBM catastrophic misses (fraud scored <0.1%): {len(lgb_catastrophic_misses)} out of {y_held_out.sum()} total fraud\n")
print(lgb_catastrophic_misses[[
    "amount", "origin_balance_error", "destination_balance_error",
    "destination_is_first_transaction", "lgb_prob", "rf_prob"
]])

LightGBM catastrophic misses (fraud scored <0.1%): 37 out of 822 total fraud

              amount  origin_balance_error  destination_balance_error  \
4786529    927729.13          0.000000e+00               0.000000e+00   
4863914    291264.20          0.000000e+00               0.000000e+00   
4870588   7445820.78          0.000000e+00               0.000000e+00   
4932995    257988.47          0.000000e+00               0.000000e+00   
4965641        63.80          0.000000e+00               6.380000e+01   
4976268    614145.31          0.000000e+00               0.000000e+00   
5025311    275082.00          0.000000e+00               0.000000e+00   
5040726    375419.35          0.000000e+00               1.000000e-02   
5133543    357226.80          0.000000e+00               0.000000e+00   
5186345    140807.76          0.000000e+00               1.000000e-02   
5187123    299016.86          0.000000e+00               0.000000e+00   
5187809       174.92          0.000000e+00    

In [26]:
print(f"Total amount at risk in these 37 LightGBM catastrophic misses: {lgb_catastrophic_misses['amount'].sum():,.0f}")

T_REVIEW = 0.0001
rf_catches_as_review = (lgb_catastrophic_misses["rf_prob"] >= T_REVIEW).sum()
print(f"Of these 37, RF flags {rf_catches_as_review} for at least REVIEW (vs. LightGBM's silent ALLOW)")

# See the full table, not truncated
pd.set_option("display.max_rows", None)
print(lgb_catastrophic_misses[["amount", "destination_is_first_transaction", "lgb_prob", "rf_prob"]])

Total amount at risk in these 37 LightGBM catastrophic misses: 48,707,865
Of these 37, RF flags 37 for at least REVIEW (vs. LightGBM's silent ALLOW)
              amount  destination_is_first_transaction       lgb_prob  \
4786529    927729.13                                 1   2.850091e-48   
4863914    291264.20                                 0   1.962889e-11   
4870588   7445820.78                                 0   2.452136e-13   
4932995    257988.47                                 0   2.251857e-05   
4965641        63.80                                 1  6.692599e-112   
4976268    614145.31                                 1   9.215818e-06   
5025311    275082.00                                 0   4.420319e-13   
5040726    375419.35                                 0   1.323744e-16   
5133543    357226.80                                 0   1.330839e-31   
5186345    140807.76                                 0   3.151649e-04   
5187123    299016.86                            

### Final Sanity Check — RF on a Broad Random Sample

Before deploying Random Forest, running the same unbiased 5,000-transaction
random validation used for LightGBM earlier, to confirm nothing outside the
specific blind-spot cases has regressed.

In [27]:
sample = X_held_out.sample(n=5000, random_state=None)
sample_labels = y_held_out.loc[sample.index]
sample_probs_rf = rf_500.predict_proba(sample)[:, 1]

T_BLOCK_RF = 0.85
T_REVIEW = 0.0001

def tier_from_prob(p, t_block):
    if p >= t_block:
        return "BLOCK"
    elif p >= T_REVIEW:
        return "REVIEW"
    return "ALLOW"

results_sample = pd.DataFrame({
    "amount": sample["amount"].values,
    "actual_fraud": sample_labels.values,
    "predicted_prob": sample_probs_rf,
    "predicted_tier": [tier_from_prob(p, T_BLOCK_RF) for p in sample_probs_rf],
})

results_sample["correctly_flagged"] = ~((results_sample["actual_fraud"] == 1) & (results_sample["predicted_tier"] == "ALLOW"))
results_sample["false_alarm"] = (results_sample["actual_fraud"] == 0) & (results_sample["predicted_tier"] != "ALLOW")

n_fraud = (results_sample["actual_fraud"] == 1).sum()
n_legit = (results_sample["actual_fraud"] == 0).sum()
print(f"RF — Fraud caught (not ALLOWed): {results_sample[results_sample['actual_fraud']==1]['correctly_flagged'].sum()} / {n_fraud}")
print(f"RF — False alarms on legit txns: {results_sample['false_alarm'].sum()} / {n_legit}")

false_alarm_rows = results_sample[results_sample["false_alarm"]]
print("\nRF false alarm tier breakdown:")
print(false_alarm_rows["predicted_tier"].value_counts())

RF — Fraud caught (not ALLOWed): 1 / 1
RF — False alarms on legit txns: 4999 / 4999

RF false alarm tier breakdown:
predicted_tier
REVIEW    4999
Name: count, dtype: int64


In [28]:
legit_probs_rf = rf_500.predict_proba(X_held_out[y_held_out == 0])[:, 1]
print("RF probability distribution on LEGITIMATE transactions:")
print(pd.Series(legit_probs_rf).describe())
print(f"\nPercentage of legit transactions with prob >= 0.0001: {(legit_probs_rf >= 0.0001).mean():.2%}")

RF probability distribution on LEGITIMATE transactions:
count    1.201820e+06
mean     5.383149e-02
std      8.932919e-02
min      1.713703e-03
25%      8.783859e-03
50%      2.781568e-02
75%      4.401642e-02
max      9.471848e-01
dtype: float64

Percentage of legit transactions with prob >= 0.0001: 100.00%


In [29]:
print("Finding RF's own cost-optimal REVIEW threshold (holding t_block=0.85):")
for t_rev in [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    c = total_cost(y_true, probs_rf, amounts, t_block=0.85, t_review=t_rev)
    c_false_alarms = ((probs_rf >= t_rev) & (probs_rf < 0.85) & (y_true == 0)).sum()
    print(f"  t_review={t_rev}: total cost = {c:,.0f}, legit txns sent to REVIEW = {c_false_alarms}")

Finding RF's own cost-optimal REVIEW threshold (holding t_block=0.85):
  t_review=0.001: total cost = 1,202,119, legit txns sent to REVIEW = 1201797
  t_review=0.01: total cost = 857,904, legit txns sent to REVIEW = 857582
  t_review=0.05: total cost = 268,122, legit txns sent to REVIEW = 267800
  t_review=0.1: total cost = 217,807, legit txns sent to REVIEW = 217485
  t_review=0.2: total cost = 57,636, legit txns sent to REVIEW = 57314
  t_review=0.3: total cost = 48,148, legit txns sent to REVIEW = 47826
  t_review=0.4: total cost = 5,540,347, legit txns sent to REVIEW = 27404
  t_review=0.5: total cost = 12,056,239, legit txns sent to REVIEW = 9434


In [30]:
print("Narrowing RF's REVIEW threshold between 0.3 and 0.4:")
for t_rev in [0.30, 0.32, 0.34, 0.35, 0.36, 0.38, 0.40]:
    c = total_cost(y_true, probs_rf, amounts, t_block=0.85, t_review=t_rev)
    pct_legit_flagged = ((probs_rf >= t_rev) & (probs_rf < 0.85) & (y_true == 0)).mean()
    n_fraud_missed = ((probs_rf < t_rev) & (y_true == 1)).sum()
    print(f"  t_review={t_rev}: cost = {c:,.0f}, legit flagged = {pct_legit_flagged:.2%}, fraud missed entirely = {n_fraud_missed}")

Narrowing RF's REVIEW threshold between 0.3 and 0.4:
  t_review=0.3: cost = 48,148, legit flagged = 3.98%, fraud missed entirely = 0
  t_review=0.32: cost = 47,547, legit flagged = 3.93%, fraud missed entirely = 0
  t_review=0.34: cost = 128,802, legit flagged = 3.79%, fraud missed entirely = 2
  t_review=0.35: cost = 239,423, legit flagged = 3.20%, fraud missed entirely = 8
  t_review=0.36: cost = 252,052, legit flagged = 3.06%, fraud missed entirely = 9
  t_review=0.38: cost = 426,075, legit flagged = 2.61%, fraud missed entirely = 16
  t_review=0.4: cost = 5,540,347, legit flagged = 2.28%, fraud missed entirely = 24


In [31]:
sample_probs_rf = rf_500.predict_proba(sample)[:, 1]  # reuse the same 5000-row sample as before

T_BLOCK_RF = 0.85
T_REVIEW_RF = 0.32

def tier_from_prob_rf(p):
    if p >= T_BLOCK_RF:
        return "BLOCK"
    elif p >= T_REVIEW_RF:
        return "REVIEW"
    return "ALLOW"

results_sample["predicted_tier_v2"] = [tier_from_prob_rf(p) for p in sample_probs_rf]
results_sample["correctly_flagged_v2"] = ~((results_sample["actual_fraud"] == 1) & (results_sample["predicted_tier_v2"] == "ALLOW"))
results_sample["false_alarm_v2"] = (results_sample["actual_fraud"] == 0) & (results_sample["predicted_tier_v2"] != "ALLOW")

print(f"RF (corrected thresholds) — Fraud caught: {results_sample[results_sample['actual_fraud']==1]['correctly_flagged_v2'].sum()} / {n_fraud}")
print(f"RF (corrected thresholds) — False alarms: {results_sample['false_alarm_v2'].sum()} / {n_legit}")
print(results_sample[results_sample['false_alarm_v2']]['predicted_tier_v2'].value_counts())

RF (corrected thresholds) — Fraud caught: 1 / 1
RF (corrected thresholds) — False alarms: 190 / 4999
predicted_tier_v2
REVIEW    190
Name: count, dtype: int64


In [39]:
joblib.dump(rf_500, r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\rf_model.pkl")

import os
print(os.path.exists(r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\rf_model.pkl"))

True


In [41]:
%whos

Variable                      Type                      Data/Info
-----------------------------------------------------------------
RandomForestClassifier        ABCMeta                   <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
SEED                          int                       42
T_BLOCK                       float                     0.9
T_BLOCK_RF                    float                     0.85
T_REVIEW                      float                     0.0001
T_REVIEW_RF                   float                     0.32
X_held_out                    DataFrame                 Shape: (1202642, 17)
X_train                       DataFrame                 Shape: (1311225, 17)
amounts                       ndarray                   1202642: 1202642 elems, type `float64`, 9621136 bytes (9.175430297851562 Mb)
average_precision_score       function                  <function average_precisi<...>re at 0x0000023A42CDBA60>
bins                          list                   

In [42]:
print('rf_500' in dir())

True


In [43]:
rf_500 = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=100,
    class_weight={0: 1, 1: 500}, random_state=SEED, n_jobs=-1,
)
rf_500.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",100
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.","{0: 1, 1: 500}"
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `m

In [44]:
joblib.dump(rf_500, r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\rf_model.pkl")

import os
print(os.path.exists(r"C:\Users\Anushka\OneDrive\Desktop\fraud-detection-project\models\rf_model.pkl"))

True


### Precision/Recall/F1 — BLOCK  (Valid Literature Comparison)

Comparing this project's strict, final auto-block decision against
Frontiers (2026)'s single-threshold result on PaySim — the correct
apples-to-apples comparison, since both represent one tuned decision
point rather than a wide net.

In [51]:
flagged_block_only = (probs_rf_full >= T_BLOCK_RF).astype(int)

precision_block = precision_score(y_true, flagged_block_only)
recall_block = recall_score(y_true, flagged_block_only)
f1_block = f1_score(y_true, flagged_block_only)

print(f"This project (RF, BLOCK only):")
print(f"  Precision: {precision_block:.4f}")
print(f"  Recall:    {recall_block:.4f}")
print(f"  F1:        {f1_block:.4f}")

This project (RF, BLOCK only):
  Precision: 0.9639
  Recall:    0.7482
  F1:        0.8425


### Cost-Ratio Sensitivity Check

Testing whether the chosen thresholds are robust to the exact cost
assumptions, since those assumptions are illustrative, not derived
from real business costs.

In [49]:
for review_cost in [1, 3, 5]:
    for block_cost in [3, 5, 10]:
        costs = []
        for i in range(len(y_true)):
            p = probs_rf_full[i]
            tier = "BLOCK" if p >= T_BLOCK_RF else ("REVIEW" if p >= T_REVIEW_RF else "ALLOW")
            if y_true[i] == 1 and tier == "ALLOW":
                costs.append(amounts[i] if 'amounts' in dir() else X_held_out['amount'].values[i])
            elif y_true[i] == 0 and tier == "BLOCK":
                costs.append(block_cost)
            elif tier == "REVIEW":
                costs.append(review_cost)
            else:
                costs.append(0)
        print(f"review_cost={review_cost}, block_cost={block_cost}: total = {sum(costs):,.0f}")

review_cost=1, block_cost=3: total = 47,501
review_cost=1, block_cost=5: total = 47,547
review_cost=1, block_cost=10: total = 47,662
review_cost=3, block_cost=3: total = 142,365
review_cost=3, block_cost=5: total = 142,411
review_cost=3, block_cost=10: total = 142,526
review_cost=5, block_cost=3: total = 237,229
review_cost=5, block_cost=5: total = 237,275
review_cost=5, block_cost=10: total = 237,390
